In [1]:
import pandas as pd

raw = pd.read_csv("../data/raw/ev_charging_sessions_raw.csv")
df = raw.copy()   # we clean the copy, never the original file

before = {
    "rows": len(raw),
    "columns": raw.shape[1],
    "missing cells": int(raw.isna().sum().sum()),
    "duplicate rows": int(raw.duplicated().sum()),
}
print(before)

{'rows': 18090, 'columns': 14, 'missing cells': 1173, 'duplicate rows': 90}


In [2]:
# Cleaning 1: remove exact duplicate rows
df = df.drop_duplicates()
print("Rows after removing duplicates:", len(df))

# Cleaning 2: tidy category text (remove extra spaces, fix capital letters)
for col in ["city", "station_type", "vehicle_type", "connector_type", "payment_method", "day_type"]:
    df[col] = df[col].str.strip()

df["station_type"] = df["station_type"].str.title()
df["vehicle_type"] = df["vehicle_type"].str.title()

print()
print(df["vehicle_type"].value_counts())
print()
print(df["station_type"].value_counts())
print()
print(df["city"].value_counts())

Rows after removing duplicates: 18000

vehicle_type
Electric Car          13517
Electric Motorbike     2066
Electric Van           1980
Electric Bus            437
Name: count, dtype: int64

station_type
Office Park           5294
Shopping Mall         4407
Fuel Station          3439
Residential Estate    3132
Highway Stop          1256
Transport Hub          472
Name: count, dtype: int64

city
Nairobi    13453
Mombasa     2466
Nakuru       788
Kisumu       768
Eldoret      525
Name: count, dtype: int64


In [3]:
import numpy as np

# Cleaning 3: fix types (text dates -> real dates)
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])

# Cleaning 4: impossible values -> mark as missing (we do not guess the true number)
print("Energy <= 0:", (df["energy_kwh"] <= 0).sum())
print("Cost <= 0:", (df["cost_kes"] <= 0).sum())
df.loc[df["energy_kwh"] <= 0, "energy_kwh"] = np.nan
df.loc[df["cost_kes"] <= 0, "cost_kes"] = np.nan

# Cleaning 5: missing categories -> "Unknown"
df["payment_method"] = df["payment_method"].fillna("Unknown")
df["connector_type"] = df["connector_type"].fillna("Unknown")

# Cleaning 6: location can be recovered, because each station has one location
station_location = df.dropna(subset=["location"]).groupby("station_id")["location"].first()
df["location"] = df["location"].fillna(df["station_id"].map(station_location))

# Cleaning 7: duration can be recalculated from the start and end times
calculated = ((df["end_time"] - df["start_time"]).dt.total_seconds() / 60).round()
df["duration_min"] = df["duration_min"].fillna(calculated)

print()
print("Rows:", len(df))
print(df.isna().sum()[df.isna().sum() > 0])

Energy <= 0: 6
Cost <= 0: 5

Rows: 18000
energy_kwh      6
cost_kes      275
dtype: int64


In [4]:
# Cleaning 8: flag energy outliers within each vehicle type (we keep the rows)
q1 = df.groupby("vehicle_type")["energy_kwh"].transform(lambda s: s.quantile(0.25))
q3 = df.groupby("vehicle_type")["energy_kwh"].transform(lambda s: s.quantile(0.75))
upper_limit = q3 + 3 * (q3 - q1)

df["energy_outlier"] = df["energy_kwh"] > upper_limit

print("Flagged outliers:", df["energy_outlier"].sum())
print()
print(df[df["energy_outlier"]].groupby("vehicle_type")["energy_kwh"].agg(["count", "max"]).round(1))
print()
print(df[df["energy_outlier"]].sort_values("energy_kwh", ascending=False)[["session_id", "vehicle_type", "energy_kwh"]].head(8))

Flagged outliers: 79

                    count    max
vehicle_type                    
Electric Bus            1  205.7
Electric Car           69  505.8
Electric Motorbike      4   26.2
Electric Van            5  334.1

       session_id  vehicle_type  energy_kwh
14284  EVS-014217  Electric Car       505.8
8069   EVS-008031  Electric Car       445.8
14476  EVS-014407  Electric Car       403.7
2237   EVS-002229  Electric Car       379.6
5551   EVS-005530  Electric Car       335.1
8219   EVS-008180  Electric Van       334.1
13161  EVS-013103  Electric Car       320.3
11256  EVS-011205  Electric Car       297.9


In [5]:
import os

os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/ev_charging_sessions_clean.csv", index=False)

after = {
    "rows": len(df),
    "columns": df.shape[1],
    "missing cells": int(df.isna().sum().sum()),
    "duplicate rows": int(df.duplicated().sum()),
}

summary = pd.DataFrame({"before": before, "after": after})
print(summary)
print()
print("Meets 15,000 rows x 10 columns:", len(df) >= 15000 and df.shape[1] >= 10)

                before  after
rows             18090  18000
columns             14     15
missing cells     1173    281
duplicate rows      90      0

Meets 15,000 rows x 10 columns: True


In [8]:
print("Sessions by city:")
sessions_by_city = df["city"].value_counts()
print(sessions_by_city)

print()
print("Sessions by station type:")
sessions_by_type = df["station_type"].value_counts()
print(sessions_by_type)

print()
print("Share of sessions in Nairobi (%):", round(sessions_by_city["Nairobi"] / len(df) * 100, 1))

Sessions by city:
city
Nairobi    13453
Mombasa     2466
Nakuru       788
Kisumu       768
Eldoret      525
Name: count, dtype: int64

Sessions by station type:
station_type
Office Park           5294
Shopping Mall         4407
Fuel Station          3439
Residential Estate    3132
Highway Stop          1256
Transport Hub          472
Name: count, dtype: int64

Share of sessions in Nairobi (%): 74.7


In [9]:
df["hour"] = df["start_time"].dt.hour

print("Sessions by hour:")
print(df["hour"].value_counts().sort_index())

print()
print("Sessions by day type:")
print(df["day_type"].value_counts())

print()
print("Busiest hour:", df["hour"].value_counts().idxmax())

Sessions by hour:
hour
0      381
1      319
2      346
3      340
4      336
5      431
6      664
7      994
8     1293
9      928
10     547
11     638
12    1027
13    1200
14     954
15     825
16     734
17     928
18    1234
19    1338
20     882
21     587
22     588
23     486
Name: count, dtype: int64

Sessions by day type:
day_type
Weekday           13032
Weekend            4546
Public Holiday      422
Name: count, dtype: int64

Busiest hour: 19


In [10]:
sessions_per_day_type = df.groupby("day_type")["session_id"].count()
days_per_day_type = df.groupby("day_type")["start_time"].apply(lambda s: s.dt.date.nunique())

fair = pd.DataFrame({
    "sessions": sessions_per_day_type,
    "days": days_per_day_type,
    "sessions_per_day": (sessions_per_day_type / days_per_day_type).round(1)
})
print(fair)

                sessions  days  sessions_per_day
day_type                                        
Public Holiday       422     6              70.3
Weekday            13032   123             106.0
Weekend             4546    52              87.4


In [11]:
duration_by_type = (
    df.groupby("station_type")["duration_min"]
    .agg(["count", "mean", "median", "max"])
    .round(1)
    .sort_values("mean", ascending=False)
)
print(duration_by_type)

                    count   mean  median    max
station_type                                   
Residential Estate   3132  242.0   221.0  720.0
Office Park          5294  220.0   205.0  720.0
Shopping Mall        4407  197.3   183.0  720.0
Highway Stop         1256  127.9    76.0  720.0
Fuel Station         3439  122.2    66.0  720.0
Transport Hub         472  119.5    96.5  720.0


In [12]:
clean_cost = df.dropna(subset=["energy_kwh", "cost_kes"])

print("Correlation between energy and cost:", round(clean_cost["energy_kwh"].corr(clean_cost["cost_kes"]), 3))
print()

clean_cost = clean_cost.assign(cost_per_kwh=clean_cost["cost_kes"] / clean_cost["energy_kwh"])
print(clean_cost.groupby("connector_type")["cost_per_kwh"].agg(["count", "mean"]).round(1))
print()
print(clean_cost.groupby("vehicle_type")[["energy_kwh", "cost_kes"]].mean().round(1))

Correlation between energy and cost: 0.854

                count  mean
connector_type             
CCS2             2964  54.6
CHAdeMO           918  52.6
GB/T              326  56.3
Type 2          13297  33.7
Unknown           214  37.7

                    energy_kwh  cost_kes
vehicle_type                            
Electric Bus              87.2    4580.4
Electric Car              22.3     847.8
Electric Motorbike         2.6      88.0
Electric Van              32.7    1393.6


In [13]:
df["fast_charger"] = df["connector_type"].isin(["CCS2", "CHAdeMO", "GB/T"])

known = df[df["connector_type"] != "Unknown"]

fast_use = known.groupby("vehicle_type")["fast_charger"].agg(["count", "sum", "mean"])
fast_use["share_fast_%"] = (fast_use["mean"] * 100).round(1)
print(fast_use[["count", "sum", "share_fast_%"]].rename(columns={"sum": "fast_sessions"}))

                    count  fast_sessions  share_fast_%
vehicle_type                                          
Electric Bus          433            433         100.0
Electric Car        13355           2953          22.1
Electric Motorbike   2038              0           0.0
Electric Van         1958            900          46.0


# Analysis: Answers to the Five Research Questions

*Note: the data is synthetic, so all findings describe this dataset only and not the real world.*

## Question 1: Which cities and station types have the most charging sessions?

**Interpretation:** In this dataset, Nairobi has the most charging sessions by far, with about 13,450 sessions, which is roughly 75% of all sessions. Office Park is the busiest station type with about 5,300 sessions, followed by Shopping Mall. Part of Nairobi's lead is because it has far more stations (28 of 48), so sessions per station would be a fairer comparison.

## Question 2: At what hours and on which day types is charging busiest?

**Interpretation:** In this dataset, the busiest hour is 19:00 with about 1,340 sessions, and the quietest is 01:00 with about 320 sessions, when there are fewer clients. There are also peaks around 08:00 and 13:00, which suggests morning, lunch-time and evening charging. Weekdays have the most sessions in total, but because there are more weekdays than other days, we compared sessions per day: weekdays still lead with about 106 sessions per day, ahead of weekend days (about 87) and public holidays (about 70, based on only 6 days).

## Question 3: Which station types have the longest sessions?

**Interpretation:** In this dataset, Residential Estates have the longest average session at about 242 minutes, followed by Office Parks at about 220 minutes, while Transport Hubs, Fuel Stations and Highway Stops are shortest at around 120 to 130 minutes. For Fuel Stations and Highway Stops the mean is much higher than the median because a few very long sessions pull the average up, so most sessions there are short. Every station type has the same maximum of 720 minutes, which is an artificial cap set in the data generator and not a real maximum.

## Question 4: How does cost change with the energy supplied?

**Interpretation:** In this dataset, cost has a strong positive correlation with energy supplied (about 0.94), meaning sessions that deliver more kWh cost more. Fast-charger connectors (CCS2, CHAdeMO and GB/T) cost about 53 to 56 KES per kWh compared with about 34 KES for Type 2, so fast charging is roughly 60% more expensive per kWh. Rows with missing cost or energy were left out of this calculation, and the flagged energy outliers are still included.

## Question 5: Which vehicle types use fast chargers most?

**Interpretation:** In this dataset, electric cars have the most fast-charger sessions in number (about 2,980) because they are the most common vehicle. As a share of their own sessions, however, buses use fast chargers most (100%), followed by vans (about 46%) and cars (about 22%). Motorbikes have no fast-charger sessions, which is expected because small batteries suit Type 2 charging, but it is also built into how the data was generated.